In [67]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from transformers import BertConfig
from tqdm import tqdm
from sklearn.metrics import f1_score
from customBERT import CustomBertModel
from Iemocap_Embeddings.embedding_loader import load_all_embeddings

In [ ]:
video_dir = "D:\IEMOCAP_full_release\Video_output_512_DINOv2"
audio_dir = "D:\IEMOCAP_full_release\Audio_output_512_HuBERT"
text_dir = "D:\IEMOCAP_full_release\Text_Embeddings"
labels_file = "D:\IEMOCAP_full_release\scene_emotions_binary.csv"
config_file = "my_custom_model\config.json"

In [39]:
video_embeddings = load_all_embeddings(video_dir)
audio_embeddings = load_all_embeddings(audio_dir)
text_embeddings = load_all_embeddings(text_dir)

In [40]:
video_embeddings = torch.from_numpy(np.stack(video_embeddings))
audio_embeddings = torch.from_numpy(np.stack(audio_embeddings))
text_embeddings = torch.from_numpy(np.stack(text_embeddings))

In [41]:
print(video_embeddings.shape)
print(audio_embeddings.shape)
print(text_embeddings.shape)

torch.Size([151, 512, 768])
torch.Size([151, 512, 768])
torch.Size([151, 512, 768])


In [42]:
df = pd.read_csv(labels_file)

# Drop scene_id column
label_array = df.drop(columns=["scene_id"]).values

labels = torch.tensor(label_array, dtype=torch.float32)

print(labels.shape)  # (151, num_classes)

torch.Size([151, 10])


In [43]:
class MultimodalDataset(Dataset):
    def __init__(self, text, video, audio, labels):
        self.text = text
        self.video = video
        self.audio = audio
        self.labels = labels

    def __len__(self):
        return self.labels.shape[0]

    def __getitem__(self, idx):
        return {
            "text": self.text[idx],       # (512,768)
            "video": self.video[idx],     # (512,768)
            "audio": self.audio[idx],     # (512,768)
            "label": self.labels[idx]     # (10,)
        }
    
dataset = MultimodalDataset(text_embeddings, video_embeddings, audio_embeddings, labels)

In [44]:
config = BertConfig.from_pretrained("my_custom_model/")
model = CustomBertModel(config)
model.load_state_dict(torch.load("my_custom_model/customBERT.bin", weights_only=True))

<All keys matched successfully>

In [45]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

CustomBertModel(
  (embeddings): BertEmbeddings(
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (subspace_proj): TripleSubspaceProjection(
    (proj1): Linear(in_features=768, out_features=256, bias=False)
    (proj2): Linear(in_features=768, out_features=256, bias=False)
    (proj3): Linear(in_features=768, out_features=256, bias=False)
  )
  (encoder): ModuleDict(
    (layer): ModuleList(
      (0): BertLayer0(
        (attention): ModuleDict(
          (self): BertSelfAttention0(
            (query): ModuleList(
              (0-11): 12 x Linear(in_features=768, out_features=64, bias=True)
            )
            (key): ModuleList(
              (0-11): 12 x Linear(in_features=768, out_features=64, bias=True)
            )
            (value): ModuleList(
              (0-11): 12 x Linear(in_features=768, out_features=64, bias=True)
            )
            (d

In [46]:
dataloader = DataLoader(
    dataset,
    batch_size=8,
    shuffle=True
)

In [54]:
with torch.no_grad():
    for batch in dataloader:

        text  = batch["text"].to(device)
        video = batch["video"].to(device)
        audio = batch["audio"].to(device)
        label = batch["label"].to(device)

        logits = model(text, video, audio)

        print(logits.shape)
        print(logits)
        print(label.shape)
        break

torch.Size([8, 10])
tensor([[ 0.1917,  0.2446, -0.2316, -0.4876,  0.0311,  0.1409, -0.3259,  0.2766,
          0.0924,  0.0958],
        [ 0.1757,  0.2066, -0.2674, -0.4691,  0.0615,  0.1440, -0.2710,  0.2625,
          0.0689,  0.0446],
        [ 0.1856,  0.2391, -0.2444, -0.4943,  0.0426,  0.1313, -0.2862,  0.2766,
          0.0798,  0.0883],
        [ 0.1799,  0.2202, -0.2342, -0.4672,  0.0141,  0.1489, -0.3069,  0.2750,
          0.0786,  0.0833],
        [ 0.1862,  0.2239, -0.2563, -0.4758,  0.0153,  0.1415, -0.2933,  0.2760,
          0.0713,  0.0875],
        [ 0.1919,  0.2011, -0.2246, -0.4632,  0.0320,  0.1621, -0.3248,  0.2667,
          0.0953,  0.0836],
        [ 0.1869,  0.2393, -0.2470, -0.4948,  0.0372,  0.1435, -0.3142,  0.2806,
          0.0841,  0.0884],
        [ 0.1845,  0.2201, -0.2316, -0.4704,  0.0311,  0.1260, -0.3170,  0.2588,
          0.0798,  0.0894]], device='cuda:0')
torch.Size([8, 10])


In [ ]:
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size],
    generator=generator
)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False,
    pin_memory=True
)

In [78]:
def train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    device,
    epochs=5,
    scheduler=None
):
    criterion = torch.nn.BCEWithLogitsLoss()

    model.to(device)

    for epoch in range(epochs):
        # -----------------------
        # Training
        # -----------------------
        model.train()
        train_loss = 0.0

        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):

            text  = batch["text"].to(device)
            video = batch["video"].to(device)
            audio = batch["audio"].to(device)
            labels = batch["label"].to(device)

            optimizer.zero_grad()

            logits = model(video, audio, text)
            loss = criterion(logits, labels)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            optimizer.step()

            if scheduler:
                scheduler.step()

            train_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)

        # -----------------------
        # Validation
        # -----------------------
        model.eval()
        val_loss = 0.0

        all_preds = []
        all_labels = []

        with torch.no_grad():
            for batch in val_loader:

                text  = batch["text"].to(device)
                video = batch["video"].to(device)
                audio = batch["audio"].to(device)
                labels = batch["label"].to(device)

                logits = model(video, audio, text)
                loss = criterion(logits, labels)

                val_loss += loss.item()
                
                probs = torch.sigmoid(logits)
                preds = (probs > 0.5).cpu()

                all_preds.append(preds)
                all_labels.append(labels.cpu())

        all_preds = torch.cat(all_preds).numpy()
        all_labels = torch.cat(all_labels).numpy()
        f1 = f1_score(all_labels, all_preds, average="micro")

        avg_val_loss = val_loss / len(val_loader)

        print(f"\nEpoch {epoch+1}")
        print(f"Train Loss: {avg_train_loss:.4f}")
        print(f"Val Loss:   {avg_val_loss:.4f}")
        print(f"Val Micro-F1: {f1:.4f}")
        print("-" * 40)

    return model

In [79]:
for param in model.parameters():
    param.requires_grad = False

for param in model.classifier.parameters():
    param.requires_grad = True

for param in model.subspace_proj.parameters():
    param.requires_grad = True

In [80]:
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

trained_model = train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    device="cuda",
    epochs=10
)

Epoch 1/10:   7%|▋         | 8/120 [00:49<11:35,  6.21s/it]


KeyboardInterrupt: 